# 07 · Two-stage stacked ensemble (X4), final model and submission

Stage 1 is the domain classifier from X2 (`D1`). Stage 2 is a gradient-boosting stacker per language whose inputs are the stage-1 domain, the validation/test outputs of every available member (B0, X1, X2, E1 with its window statistics, X3), the raw LM features and text statistics.

The stacker only sees validation rows: it is scored with out-of-fold predictions from a 5-fold `StratifiedGroupKFold` over the validation split (groups from `02_split`), so it is never evaluated on rows it was fitted on; for the test set it is refit on all validation rows. Members missing on disk are skipped.

The final model is the one with the highest validation macro-F1 in the experiment log.

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

from hw3.config import ROOT, art, load_config

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.width", 200, "display.max_columns", 40)
FIG = ROOT / "images"

from hw3 import ensemble
from hw3.runs import compare, experiment_log, report
from hw3.submission import save_submission, select_final

In [ ]:
Xv, oof = ensemble.run_ensemble(load_config("ensemble"))
meta = json.loads(art("runs", "X4.json").read_text())
meta["members"], Xv.shape

In [ ]:
rep = report("X4")
rep["figure"].savefig(FIG / "confusion_X4.png", dpi=150, bbox_inches="tight")
rep["per_class"]

In [ ]:
compare(["B0", "X1", "X2", "E1", "X3", "X4"])

## Final model selection

In [ ]:
log = experiment_log()
best = select_final(log)
log

In [ ]:
best

## Submission with automated checks

The checks run on the frame before writing and again on the file read back from disk; any failure raises and nothing is uploaded.

In [ ]:
save_submission(best.id)

Upload manually after the checks pass:

```bash
kaggle competitions submit -c ai-generated-text-detection-hw3-ucu-ai-2026 -f submission.csv -m "<model id>"
```